In [0]:
path = r'/Volumes/my_project/e-commerce/bronze/order_items_final.csv'
order_items_final_df = spark.read.format('csv').option("header", "true").option("inferSchema", "true").load(path)
order_items_final_df.display()

Return from customer table to order item final table as it is primary key

In [0]:
import sys
sys.path.append("/Workspace/Repos/aasthapetkar2003@gmail.com/customers_utils")

from pk_column_clean import clean_customer_id

order_items_final_df = clean_customer_id(order_items_final_df)
display(order_items_final_df)

Return from customer table to sort the order id, customer id,product id, seller id

In [0]:
from pk_column_clean import clean_id_columns_fk

id_columns = ["order_id", "customer_id", "product_id", "seller_id"]
order_items_final_df = clean_id_columns_fk(order_items_final_df, id_columns)
order_items_final_df.display()

Return the status data from inventory table

In [0]:
from pk_column_clean import clean_status_column

order_items_final_df = clean_status_column(order_items_final_df)
order_items_final_df.display()

Cleaning the data of amount

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

order_items_final_df = (
    order_items_final_df

    # STEP 1: Normalize ALL whitespace (spaces, tabs, NBSP)
    .withColumn(
        "amount",
        F.regexp_replace(F.col("amount").cast("string"), r"[\s\u00A0]+", "")
    )

    # STEP 2: Remove everything except digits and dot
    .withColumn(
        "amount",
        F.regexp_replace(F.col("amount"), r"[^0-9.]", "")
    )

    # STEP 3: Remove leading zeros
    .withColumn(
        "amount",
        F.regexp_replace(F.col("amount"), r"^0+(?=\d)", "")
    )

    # STEP 4: Convert to decimal if valid
    .withColumn(
        "amount",
        F.when(
            F.col("amount").rlike(r"^\d+(\.\d+)?$"),
            F.col("amount").cast(DecimalType(18, 2))
        ).otherwise(F.lit(None))
    )

    # STEP 5: Flag invalid/unrealistic values
    .withColumn(
        "amount_valid_flag",
        F.when(
            F.col("amount").isNull() |
            (F.col("amount") < 0) |
            (F.col("amount") > 100000000),
            "N"
        ).otherwise("Y")
    )
)

order_items_final_df.display()

Return the quantity data from inventory table

In [0]:
from pk_column_clean import clean_quantity_column

order_items_final_df = clean_quantity_column(order_items_final_df)
order_items_final_df.display()

Return the Data of city from Customer table

In [0]:
from pk_column_clean import clean_city_column

order_items_final_df = clean_city_column(order_items_final_df)
order_items_final_df.display()

Return the data of state from customer table

In [0]:
from pk_column_clean import clean_state_column

order_items_final_df = clean_state_column(order_items_final_df, 'state')
order_items_final_df.display()

Return the data of email from customer table

In [0]:
from pk_column_clean import clean_email_column

order_items_final_df = clean_email_column(order_items_final_df, 'email')
order_items_final_df.display()

Return the data of mobile number from customer table

In [0]:
from pk_column_clean import clean_mobile_number_column

order_items_final_df = clean_mobile_number_column(order_items_final_df)
order_items_final_df.display()

Return the data of created_date and updated_date from customer table

In [0]:
from pk_column_clean import clean_created_at_column

order_items_final_df = clean_created_at_column(order_items_final_df)
order_items_final_df.display()

In [0]:
from pk_column_clean import clean_updated_at_column

order_items_final_df = clean_updated_at_column(order_items_final_df)
order_items_final_df.display()

Return the value of remark from inventory table

In [0]:
from pk_column_clean import clean_remarks_column

order_items_final_df = clean_remarks_column(order_items_final_df)
order_items_final_df.display()

Return the data of source_sytem from customer table

In [0]:
from pk_column_clean import clean_source_system_column

order_items_final_df = clean_source_system_column(order_items_final_df)
order_items_final_df.display()

Return the data of rating from inventory table

In [0]:
from pk_column_clean import clean_rating_column

order_items_final_df = clean_rating_column(order_items_final_df)
order_items_final_df.display()

Return the data of event_date from inventory table

In [0]:
from pk_column_clean import clean_event_date_column

order_items_final_df = clean_event_date_column(order_items_final_df)
order_items_final_df.display()

writing order_item_df to silver layer in delta format

In [0]:
silver_path=r"/Volumes/my_project/e-commerce/silver/Order_Item_final"

order_items_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)
